# V21 confirmatory unit muril_t43 (frozen at tag v21-freeze; docs/PROJECT_STATE.md, V21 pre-registration). TEST split: XNLI-en + IndicXNLI, 11 languages.

In [ ]:
%%bash
set -e
exec > >(tee -a /kaggle/working/run.log) 2>&1
M=muril; S=43; UNIT=${M}_t${S}; MID=google/muril-base-cased
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v21-freeze && git log --oneline -1 | tee /kaggle/working/commit.txt
O=/kaggle/working/v21; mkdir -p $O
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | tee $O/gpu.txt
pip install -q -r requirements-ci.txt && pip install -q -e . --no-deps
pip uninstall -y -q torchvision torchaudio || true
python scripts/preflight.py check | tee $O/preflight.txt
python -c "import transformers,torch;assert transformers.__version__=='4.51.3' and torch.__version__.startswith('2.6.0');print(torch.__version__,transformers.__version__)" | tee $O/env.txt
# a new teacher: the V20 recipe unchanged, fine-tuning seed $S; the gate is checked on validation only (never replaced)
FT=$O/ft
python scripts/v20_finetune.py --out $FT --model-id $MID --seed $S > $O/ft.log 2>&1 || { tail -30 $O/ft.log; exit 1; }
grep -v -i warn $O/ft.log | tail -16
grep -q '"gate_pass": true' $FT/ft_meta.json || { echo TEACHER_GATE_FAIL; exit 1; }
# E1-verified local tokenizer dirs; the TEST evaluation sets (XNLI-en + IndicXNLI, 11 languages) for the add-on
PYTHONPATH=scripts python scripts/v21_e1_prep.py --out /tmp/v21in --tok-only
PYTHONPATH=scripts python scripts/v21_run.py --ckpt $FT/mnli --unit $UNIT --write-extra-eval /tmp/v21in/extra_test.json --allow-test
# official K-prune + Kwon: English calibration (the V20 index files), c 0.5, seeds 7 42 77, students predict on the test sets
export MODEL_NAME=v21tok_$M BL_SKIP_CKPT=1 BL_TASKS=mnli BL_CONS=0.5 PREP_ARGS=--no-ckpt CKPT_DIR_ROOT=$FT
export BL_KWON_SEEDS="7 42 77" BL_KPRUNE_SEEDS="7 42 77" V21_TOK_DIR=/tmp/v21in/tok V21_EXTRA_EVAL=/tmp/v21in/extra_test.json
bash scripts/v15_kaggle.sh bl /tmp/bl $O > $O/bl_stage.log 2>&1 || echo BL_FAILED
unset V21_EXTRA_EVAL MODEL_NAME
grep -i "FAIL\|Traceback" $O/bl_stage.log | cut -c1-200 | tail -20 || true
# our conditions + official files + per-unit integrity gate (accuracies stay in the files; the analysis runs once, later)
PYTHONPATH=scripts python scripts/v21_run.py --ckpt $FT/mnli --unit $UNIT --kprune-masks $O/bl/kprune_masks.npz \
  --index-dir /tmp/bl/inputs/indices --official-root /tmp/bl/bl --ft-meta $FT/ft_meta.json --output-dir $O/units --allow-test \
  > $O/run_$UNIT.log 2>&1 || { grep -v -i "warn\|xet\|acc=" $O/run_$UNIT.log | tail -30; echo RUN_FAILED; }
grep "integrity" $O/run_$UNIT.log || true
cp /kaggle/working/run.log $O/run.log; ls -la $O $O/units/$UNIT || true
